# 07 — Final comparison

Headline figures for the report. Reads `results/depth_sweep.json`,
`results/size_scaling.json`, `results/risk_sweep.json` — no heavy
computation, just plotting. Run notebooks 04, 05, 06 first.

In [1]:
# === Bootstrap (Colab + local) ===
import os, urllib.request as _u
exec((open('../scripts/bootstrap.py') if os.path.exists('../scripts/bootstrap.py') else _u.urlopen('https://raw.githubusercontent.com/egil10/fys5419/main/project2/code/scripts/bootstrap.py')).read())

# === Project imports ===
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scripts.colab    import out_dir
from scripts.plotting import apply_style, PALETTE, title, fig_path
apply_style()

RESULTS = out_dir('results')
print(f'Reading sweep results from: {RESULTS}')

> /usr/bin/python3 -m pip install -q numpy pandas scipy matplotlib yfinance pyarrow
[colab.setup] env=Colab
[colab.setup] cwd  = /content/fys5419/project2/code/notebooks
[colab.setup] path = /content/fys5419/project2/code  (added to sys.path)
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Reading sweep results from: /content/drive/MyDrive/GITHUB-COLAB/fys5419/project2/code/results


### Figure 1 — Depth sweep (Sweep 1)

In [ ]:
sweep = pd.DataFrame(json.loads((RESULTS / 'depth_sweep.json').read_text()))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(sweep.p, sweep.ratio, 'o-', color=PALETTE['blue'], lw=2, ms=8)
axes[0].axhline(1.0, color=PALETTE['red'], ls='--', lw=1.2, label='optimum (r=1)')
axes[0].set_xticks(sweep.p); axes[0].set_xlabel('depth p')
axes[0].set_ylabel(r'scaled ratio $(E_{worst} - E)/(E_{worst} - E_{opt})$')
axes[0].set_ylim(0.0, 1.05)
title(axes[0], 'QAOA approximation ratio (scaled)', '1.0 = ground state; ~0.5 = unbiased')
axes[0].legend()
axes[0].grid(False)

axes[1].plot(sweep.p, sweep.p_optimal,  'o-', color=PALETTE['red'],         lw=2, ms=8, label='P(optimum)')
axes[1].plot(sweep.p, sweep.p_feasible, 'o-', color=PALETTE['blue_muted'], lw=2, ms=8, label='P(feasible)')
axes[1].axhline(1.0/(1 << 16), color=PALETTE['charcoal'], ls=':', lw=1, label=r'uniform $1/2^{16}$')
axes[1].set_xticks(sweep.p); axes[1].set_xlabel('depth p'); axes[1].set_ylabel('probability')
title(axes[1], 'Measurement probabilities', 'higher = more concentrated on the right answer')
axes[1].legend()
axes[1].grid(False)

plt.tight_layout()
fig.savefig(fig_path('compare', 'depth_sweep'), bbox_inches='tight')
plt.show()

### Figure 2 — Size scaling (Sweep 2)

In [ ]:
# Size-scaling sweep is now produced as one record per (n, solver, subset).
# Aggregate to median + IQR across subsets for each (n, solver).
#
# 'ratio' is the SCALED approximation ratio in [0, 1] — see 05_scaling.ipynb
# for the definition. If your cache predates that change you'll see ratios
# of -2846 etc.; delete results/size_scaling.json and rerun 05 to refresh.
scale = pd.DataFrame(json.loads((RESULTS / 'size_scaling.json').read_text()))

if scale['ratio'].min() < -0.5 or scale['ratio'].max() > 1.5:
    print(f'WARN: size_scaling.json has ratios in '
          f'[{scale.ratio.min():.1f}, {scale.ratio.max():.1f}] — looks like '
          f'the legacy cost/E_opt format. Plots below will be unreadable. '
          f'Delete the cache and rerun 05_scaling to fix.')

agg = (scale
       .groupby(['n', 'solver'])
       .agg(median_ratio=('ratio', 'median'),
            q25_ratio   =('ratio', lambda s: s.quantile(0.25)),
            q75_ratio   =('ratio', lambda s: s.quantile(0.75)),
            median_rt   =('runtime_s', 'median'),
            n_subsets   =('ratio', 'count'))
       .reset_index())

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for solver, sub in agg.groupby('solver'):
    sub = sub.sort_values('n')
    line, = axes[0].plot(sub.n, sub.median_ratio, 'o-', lw=2, ms=7, label=solver)
    axes[0].fill_between(sub.n, sub.q25_ratio, sub.q75_ratio,
                         color=line.get_color(), alpha=0.18)
    axes[1].semilogy(sub.n, sub.median_rt, 'o-', lw=2, ms=7,
                     color=line.get_color(), label=solver)

axes[0].axhline(1.0, color=PALETTE['charcoal'], ls=':', lw=1, label='optimum')
axes[0].set_xlabel('problem size n')
axes[0].set_ylabel(r'scaled ratio $(E_{worst} - E)/(E_{worst} - E_{opt})$')
axes[0].set_ylim(0.0, 1.05)
title(axes[0], 'Quality vs problem size',
      'median +/- IQR over random size-n subsets; 1.0 = brute-force optimum')
axes[0].legend(fontsize=8)
axes[0].grid(False)

axes[1].set_xlabel('problem size n'); axes[1].set_ylabel('runtime (s, log)')
title(axes[1], 'Runtime vs problem size', 'median across subsets')
axes[1].legend(fontsize=8)
axes[1].grid(False)

plt.tight_layout()
fig.savefig(fig_path('compare', 'scaling'), bbox_inches='tight')
plt.show()

### Figure 3 — Risk landscape (Sweep 3)

In [ ]:
risk = pd.DataFrame(json.loads((RESULTS / 'risk_sweep.json').read_text()))

if risk['ratio'].min() < -0.5 or risk['ratio'].max() > 1.5:
    print(f'WARN: risk_sweep.json has ratios in '
          f'[{risk.ratio.min():.1f}, {risk.ratio.max():.1f}] — looks like '
          f'the legacy cost/E_opt format. Delete the cache and rerun 06_risk.')

fig, ax = plt.subplots(figsize=(10, 5))
for solver, sub in risk.groupby('solver'):
    ax.semilogx(sub['lambda'], sub.ratio, 'o-', lw=2, ms=7, label=solver)
ax.axhline(1.0, color=PALETTE['charcoal'], ls=':', lw=1, label='optimum')
ax.set_xlabel(r'risk aversion $\lambda$ (log)')
ax.set_ylabel(r'scaled ratio $(E_{worst} - E)/(E_{worst} - E_{opt})$')
ax.set_ylim(0.0, 1.05)
title(ax, 'Approximation quality vs risk aversion',
      r'low $\lambda$: near-linear (greedy wins); high $\lambda$: frustrated (QAOA mechanism matters)')
ax.legend(fontsize=9)
ax.grid(False)
plt.tight_layout()
fig.savefig(fig_path('compare', 'risk'), bbox_inches='tight')
plt.show()